<a href="https://colab.research.google.com/github/yo-danny/speech-emotion-recognition/blob/main/speech_emotion_recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Speech Emotion Recognition



In [1]:
import librosa
import soundfile
import os, glob, pickle
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

Defining a function to extract mfcc, chroma and mel features from the audio.

Being:

- mfcc: Mel Frequency Cepstral Coefficient, represents the short-term power spectrum of a sound
- chroma: Pertains to the 12 different pitch classes
- mel: Mel Spectrogram Frequency

In [13]:
def extract_feature (file_name, mfcc, chroma, mel):
    with soundfile.SoundFile(file_name) as sound_file:
        X = sound_file.read(dtype="float32")
        sample_rate=sound_file.samplerate
        if chroma:
            stft = np.abs(librosa.stft(X))
            chroma_mean=np.mean(librosa.feature.chroma_stft(S=stft, sr=sample_rate).T, axis=0)
            result = np.array([])
            np.hstack((result, chroma_mean))
        if mfcc:
            mfcc_mean=np.mean(librosa.feature.mfcc(y=X, sr=sample_rate, n_mfcc=40).T, axis=0)
            np.hstack((result, mfcc_mean))
        if mel:
            mel_mean=np.mean(librosa.feature.melspectrogram(y=X, sr=sample_rate).T,axis=0)
            np.hstack((result, mel_mean))
    return result

Dictionary to hold number values for emotions avaible in the RAVDESS dataset.

In [3]:
emotions = {
    '01': 'neutral',
    '02': 'calm',
    '03': 'happy',
    '04': 'sad',
    '05': 'angry',
    '06': 'fearful',
    '07': 'disgust',
    '08': 'surprised'
}

Function to load the data from our ambient, calls the extraction of features and return the train and test datasets for model training.

In [7]:
def load_data(test_size=0.2):
    x, y = [], []
    for file in glob.glob("/content/drive/MyDrive/speech-emotion-recognition-ravdess-data/Actor_*/*.wav"):
        file_name = os.path.basename(file)
        emotions = file_name.split("-")[2]
        feature = extract_feature(file_name=file, mfcc=True, chroma=True, mel=True)
        x.append(feature)
        y.append(emotions)
    return train_test_split(np.array(x), y, test_size=test_size, random_state=9)


In [ ]:
x_train, x_test, y_train, y_test = load_data(test_size=0.25)

# Get the shape of the training and testing datasets
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

print(f"Number of extracted features: {x_train.shape[1]}")

Initializing the MLPClassifier.

In [ ]:
model = MLPClassifier(alpha=0.01, batch_size=256, epsilon=1e-08, hidden_layer_sizes=(300,), learning_rate='adaptive', max_iter=500)

model.fit(x_train, y_train)

In [ ]:
y_pred = model.predict(x_test)

accuracy=accuracy_score(y_true=y_test, y_pred=y_pred)

print("Accuracy: {:.2f}%".format(accuracy*100))